## Spark Session Setup

Initializes Apache Spark and configures the environment required for processing the fraud detection dataset.

In [1]:
# Import required libraries

import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark = SparkSession.builder \
    .appName("FraudPreprocessing") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate()


print("Spark Version:", spark.version)

Spark Version: 3.5.9


## Load Dataset

Reads the PaySim transaction dataset into a Spark DataFrame for further processing.

In [2]:
from pathlib import Path


project_root = Path(
    r"C:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline"
)


csv_path = project_root / "data" / "PS_20174392719_1491204439457_log.csv"


fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)


print("Dataset loaded successfully")
print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

Dataset loaded successfully
Rows: 6362620
Columns: 11


## 2. Remove Unnecessary Columns

The columns `nameOrig` and `nameDest` represent transaction identifiers.

They do not provide meaningful patterns for fraud prediction.

`isFlaggedFraud` is removed because it is a system-generated flag and may introduce bias.

In [3]:
processed_df = fraud_df.drop(
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
)


print("Remaining Columns:")
processed_df.columns

Remaining Columns:


['step',
 'type',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud']

## 3. Feature Engineering

Additional features are created to capture suspicious transaction behaviour.

### Origin Balance Change

Difference between the original account balance before and after transaction.

### Destination Balance Change

Difference between destination account balance before and after transaction.

These features help identify abnormal money movement patterns.

In [4]:
processed_df = processed_df.withColumn(
    "orig_balance_change",
    col("oldbalanceOrg") - col("newbalanceOrig")
)


processed_df = processed_df.withColumn(
    "dest_balance_change",
    col("newbalanceDest") - col("oldbalanceDest")
)


processed_df.show(5)

+----+--------+--------+-------------+--------------+--------------+--------------+-------+-------------------+-------------------+
|step|    type|  amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|orig_balance_change|dest_balance_change|
+----+--------+--------+-------------+--------------+--------------+--------------+-------+-------------------+-------------------+
|   1| PAYMENT| 9839.64|     170136.0|     160296.36|           0.0|           0.0|      0|  9839.640000000014|                0.0|
|   1| PAYMENT| 1864.28|      21249.0|      19384.72|           0.0|           0.0|      0| 1864.2799999999988|                0.0|
|   1|TRANSFER|   181.0|        181.0|           0.0|           0.0|           0.0|      1|              181.0|                0.0|
|   1|CASH_OUT|   181.0|        181.0|           0.0|       21182.0|           0.0|      1|              181.0|           -21182.0|
|   1| PAYMENT|11668.14|      41554.0|      29885.86|           0.0|        

## 4. Convert Transaction Type into Numerical Values

Machine learning algorithms cannot directly process categorical values.

The transaction type column contains values such as:

- CASH_OUT
- PAYMENT
- TRANSFER
- CASH_IN
- DEBIT

StringIndexer converts these categories into numerical representations.

In [5]:
from pyspark.ml.feature import StringIndexer


type_indexer = StringIndexer(
    inputCol="type",
    outputCol="type_index"
)


processed_df = type_indexer.fit(processed_df).transform(processed_df)


processed_df.select(
    "type",
    "type_index"
).show(10)

+--------+----------+
|    type|type_index|
+--------+----------+
| PAYMENT|       1.0|
| PAYMENT|       1.0|
|TRANSFER|       3.0|
|CASH_OUT|       0.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
|   DEBIT|       4.0|
+--------+----------+
only showing top 10 rows



## 5. Verify Processed Dataset

Check the final schema after preprocessing.

In [6]:
processed_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- orig_balance_change: double (nullable = true)
 |-- dest_balance_change: double (nullable = true)
 |-- type_index: double (nullable = false)



## 6. Create Feature Vector

Combines all input features into a single vector column required by Spark ML algorithms.

In [7]:
from pyspark.ml.feature import VectorAssembler


feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "type_index"
]


assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)


final_df = assembler.transform(processed_df)


final_df.select(
    "features",
    "isFraud"
).show(5)

+--------------------+-------+
|            features|isFraud|
+--------------------+-------+
|[1.0,9839.64,1701...|      0|
|[1.0,1864.28,2124...|      0|
|[1.0,181.0,181.0,...|      1|
|[1.0,181.0,181.0,...|      1|
|[1.0,11668.14,415...|      0|
+--------------------+-------+
only showing top 5 rows



## 7. Select Required Columns

Keeps only the feature vector and fraud label required for model training.

In [8]:
model_df = final_df.select(
    "features",
    col("isFraud").alias("label")
)


model_df.show(5)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[1.0,9839.64,1701...|    0|
|[1.0,1864.28,2124...|    0|
|[1.0,181.0,181.0,...|    1|
|[1.0,181.0,181.0,...|    1|
|[1.0,11668.14,415...|    0|
+--------------------+-----+
only showing top 5 rows



## 8. Check Final Dataset

Displays the final dataset structure before splitting into training and testing data.

In [9]:
model_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)



## 9. Split Dataset into Training and Testing Data

Divides the processed dataset into training and testing sets.

The training data is used to build the fraud detection model, while the testing data is used to evaluate model performance.

In [10]:
# Split dataset into train and test sets

train_df, test_df = model_df.randomSplit(
    [0.8, 0.2],
    seed=42
)


print("Training Data Count:", train_df.count())
print("Testing Data Count:", test_df.count())

Training Data Count: 5089858
Testing Data Count: 1272762


## 10. Check Fraud Distribution in Training Data

Checks the number of fraudulent and normal transactions available in the training dataset.

In [11]:
train_df.groupBy("label") \
    .count() \
    .show()

+-----+-------+
|label|  count|
+-----+-------+
|    0|5083312|
|    1|   6546|
+-----+-------+



## 11. Check Fraud Distribution in Testing Data

Checks whether the testing dataset contains both fraud and legitimate transactions.

In [12]:
test_df.groupBy("label") \
    .count() \
    .show()

+-----+-------+
|label|  count|
+-----+-------+
|    0|1271095|
|    1|   1667|
+-----+-------+



## Data Preparation Completed

The transaction dataset has been successfully transformed into a machine learning-ready format.

The final dataset contains:
- Engineered transaction features
- Encoded transaction types
- Feature vectors
- Training and testing datasets

The prepared data will be used for fraud detection model training.

In [13]:
print("Spark preprocessing completed successfully!")

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

Spark preprocessing completed successfully!


Training records: 5089858
Testing records: 1272762


In [14]:
spark.stop()